In [1]:
# =========================
# CryptoPunks Loader (Colab)
# =========================

import random
import time
import sqlite3
from dataclasses import dataclass
from typing import Optional, Tuple, List

import requests
from bs4 import BeautifulSoup
from dateutil.parser import parse as parse_date


# -------------------------
# CONFIG — EDIT THESE
# -------------------------

START_PUNK = 7879      # inclusive
END_PUNK = 8181      # inclusive (keep small in Colab first)
DB_NAME = "punkDB.db"

# polite scraping
DELAY_MIN = 1.5       # seconds
DELAY_MAX = 3.0
TIMEOUT = 20.0
MAX_RETRIES = 3
BACKOFF_BASE = 1.5

# -------------------------
# CONSTANTS / MAPS
# -------------------------
#BASE_URL = "https://www.cryptopunks.app/cryptopunks/details/"
BASE_URL = "https://www.larvalabs.com/cryptopunks/details/"

punktype_ids = {
    "Alien": 0,
    "Ape": 1,
    "Zombie": 2,
    "Female": 3,
    "Male": 4,
}

attribute_ids = {
    "Beanie": 0, "Choker": 1, "Pilot Helmet": 2, "Tiara": 3,
    "Orange Side": 4, "Buck Teeth": 5, "Welding Goggles": 6,
    "Pigtails": 7, "Pink With Hat": 8, "Top Hat": 9,
    "Spots": 10, "Rosy Cheeks": 11, "Blonde Short": 12,
    "Wild White Hair": 13, "Cowboy Hat": 14, "Wild Blonde": 15,
    "Straight Hair Blonde": 16, "Big Beard": 17, "Red Mohawk": 18,
    "Half Shaved": 19, "Blonde Bob": 20, "Vampire Hair": 21,
    "Clown Hair Green": 22, "Straight Hair Dark": 23,
    "Straight Hair": 24, "Silver Chain": 25, "Dark Hair": 26,
    "Purple Hair": 27, "Gold Chain": 28, "Medical Mask": 29,
    "Tassle Hat": 30, "Fedora": 31, "Police Cap": 32,
    "Clown Nose": 33, "Smile": 34, "Cap Forward": 35,
    "Hoodie": 36, "Front Beard Dark": 37, "Frown": 38,
    "Purple Eye Shadow": 39, "Handlebars": 40, "Blue Eye Shadow": 41,
    "Green Eye Shadow": 42, "Vape": 43, "Front Beard": 44,
    "Chinstrap": 45, "3D Glasses": 46, "Luxurious Beard": 47,
    "Mustache": 48, "Normal Beard Black": 49, "Normal Beard": 50,
    "Eye Mask": 51, "Goat": 52, "Do-rag": 53, "Shaved Head": 54,
    "Muttonchops": 55, "Peak Spike": 56, "Pipe": 57, "VR": 58,
    "Cap": 59, "Small Shades": 60, "Clown Eyes Green": 61,
    "Clown Eyes Blue": 62, "Headband": 63, "Crazy Hair": 64,
    "Knitted Cap": 65, "Mohawk Dark": 66, "Mohawk": 67,
    "Mohawk Thin": 68, "Frumpy Hair": 69, "Wild Hair": 70,
    "Messy Hair": 71, "Eye Patch": 72, "Stringy Hair": 73,
    "Bandana": 74, "Classic Shades": 75, "Shadow Beard": 76,
    "Regular Shades": 77, "Horned Rim Glasses": 78,
    "Big Shades": 79, "Nerd Glasses": 80,
    "Black Lipstick": 81, "Mole": 82, "Purple Lipstick": 83,
    "Hot Lipstick": 84, "Cigarette": 85, "Earring": 86,
}

# -------------------------
# DB HELPERS
# -------------------------

def connect_db(db):
    con = sqlite3.connect(db)
    con.execute("PRAGMA journal_mode=WAL;")
    return con


def init_db(con):
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS Punks(
            PunkID INTEGER PRIMARY KEY,
            PunkType INTEGER,
            AttributeVec TEXT
        );
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS PunkTrades(
            TDate INTEGER,
            PunkID INTEGER,
            TType TEXT,
            TFrom TEXT,
            TTo TEXT,
            TAmt INTEGER,
            PRIMARY KEY (TDate, PunkID, TType, TFrom, TTo, TAmt)
        );
    """)

    cur.execute("CREATE INDEX IF NOT EXISTS idx_trades_punk ON PunkTrades(PunkID);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_trades_amt ON PunkTrades(TAmt);")
    con.commit()


# -------------------------
# SCRAPING HELPERS
# -------------------------

def polite_sleep():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))


def fetch_html(session, url):
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = session.get(url, timeout=TIMEOUT)
            r.raise_for_status()
            return r.text
        except Exception:
            time.sleep(BACKOFF_BASE * (2 ** attempt))
    raise RuntimeError(f"Failed to fetch {url}")


def parse_punk(soup):
    vec = ["0"] * 87

    for div in soup.find_all("div", class_="col-md-4"):
        a = div.find("a")
        if a and a.text in attribute_ids:
            vec[attribute_ids[a.text]] = "1"

    punk_type = -1
    for h4 in soup.find_all("h4"):
        a = h4.find("a")
        if a and a.text in punktype_ids:
            punk_type = punktype_ids[a.text]
            break

    return punk_type, "".join(vec)


def parse_amount(txt):
    try:
        s = txt[txt.index("$")+1:txt.index(")")]
        if s.endswith("M"):
            return int(float(s[:-1].replace(",", "")) * 1e6)
        return int(s.replace(",", ""))
    except:
        return 0


def parse_trades(soup, punk_id):
    rows = []
    table = soup.find("table", class_="table")
    if not table:
        return rows

    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 5:
            continue

        ttype = tds[0].text.strip()
        tfrom = tds[1].text.strip()
        tto = tds[2].text.strip()
        amt = parse_amount(tds[3].text)
        try:
            date = int(parse_date(tds[4].text).strftime("%Y%m%d"))
        except:
            continue

        rows.append((date, punk_id, ttype, tfrom, tto, amt))

    return rows


# -------------------------
# SUMMARIES
# -------------------------

def summaries(con):
    cur = con.cursor()

    print("\n=== SUMMARIES ===\n")

    cur.execute("""
        SELECT PunkID, TAmt FROM PunkTrades
        WHERE TType='Sold' AND TAmt>0
        ORDER BY TAmt DESC LIMIT 1
    """)
    r = cur.fetchone()
    if r:
        print(f"Most expensive punk: #{r[0]} sold for ${r[1]:,}")

    cur.execute("""
        SELECT PunkID, COUNT(*) FROM PunkTrades
        GROUP BY PunkID ORDER BY COUNT(*) DESC LIMIT 1
    """)
    r = cur.fetchone()
    print(f"Most traded punk: #{r[0]} with {r[1]} events")

    cur.execute("""
        SELECT PunkID, SUM(TAmt) FROM PunkTrades
        WHERE TType='Sold' AND TAmt>0
        GROUP BY PunkID ORDER BY SUM(TAmt) DESC LIMIT 1
    """)
    r = cur.fetchone()
    print(f"Highest total USD volume: #{r[0]} = ${r[1]:,}")


# -------------------------
# MAIN (COLAB SAFE)
# -------------------------

def main():
    print(f"Scraping punks {START_PUNK} → {END_PUNK}")
    con = connect_db(DB_NAME)
    init_db(con)

    session = requests.Session()
    session.headers["User-Agent"] = "Mozilla/5.0"

    total = END_PUNK - START_PUNK + 1

    for i, punk_id in enumerate(range(START_PUNK, END_PUNK + 1), 1):
        print(f"[{i}/{total}] Punk {punk_id}")

        html = fetch_html(session, BASE_URL + str(punk_id))
        soup = BeautifulSoup(html, "html.parser")

        punk_type, vec = parse_punk(soup)
        trades = parse_trades(soup, punk_id)

        with con:
            con.execute(
                "INSERT OR REPLACE INTO Punks VALUES(?,?,?)",
                (punk_id, punk_type, vec)
            )
            con.executemany(
                "INSERT OR IGNORE INTO PunkTrades VALUES(?,?,?,?,?,?)",
                trades
            )

        polite_sleep()

    summaries(con)
    con.close()
    print("\nDone. DB saved as", DB_NAME)


# ---- RUN ----
main()


Scraping punks 7879 → 8181
[1/303] Punk 7879
[2/303] Punk 7880
[3/303] Punk 7881
[4/303] Punk 7882
[5/303] Punk 7883
[6/303] Punk 7884
[7/303] Punk 7885
[8/303] Punk 7886
[9/303] Punk 7887
[10/303] Punk 7888
[11/303] Punk 7889
[12/303] Punk 7890
[13/303] Punk 7891
[14/303] Punk 7892
[15/303] Punk 7893
[16/303] Punk 7894
[17/303] Punk 7895
[18/303] Punk 7896
[19/303] Punk 7897
[20/303] Punk 7898
[21/303] Punk 7899
[22/303] Punk 7900
[23/303] Punk 7901
[24/303] Punk 7902
[25/303] Punk 7903
[26/303] Punk 7904
[27/303] Punk 7905
[28/303] Punk 7906
[29/303] Punk 7907
[30/303] Punk 7908
[31/303] Punk 7909
[32/303] Punk 7910
[33/303] Punk 7911
[34/303] Punk 7912
[35/303] Punk 7913
[36/303] Punk 7914
[37/303] Punk 7915
[38/303] Punk 7916
[39/303] Punk 7917
[40/303] Punk 7918
[41/303] Punk 7919
[42/303] Punk 7920
[43/303] Punk 7921
[44/303] Punk 7922
[45/303] Punk 7923
[46/303] Punk 7924
[47/303] Punk 7925
[48/303] Punk 7926
[49/303] Punk 7927
[50/303] Punk 7928
[51/303] Punk 7929
[52/303] Punk

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("punkDB.db")
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
pd.read_sql("SELECT COUNT(*) AS total_punks FROM Punks;", conn)
pd.read_sql("SELECT * FROM Punks LIMIT 5;", conn)
pd.read_sql("SELECT * FROM PunkTrades LIMIT 10;", conn)


,TDate,PunkID,TType,TFrom,TTo,TAmt
0,20250124,7879,Bid Withdrawn,0xb4ea05,,81
1,20250124,7879,Bid,0xb4ea05,,81
2,20241229,7879,Bid,0x86ff70,,0
3,20210805,7879,Transfer,0xb275e5,0x20744f,0
4,20210728,7879,Sold,0x577ebc,0xb275e5,54929
5,20210510,7879,Offered,,,97901
6,20210503,7879,Offer Withdrawn,,,0
7,20210429,7879,Offered,,,73078
8,20210131,7879,Transfer,0xcbd482,0x577ebc,0
9,20170623,7879,Claimed,,0xcbd482,0
